In [0]:

from pathlib import Path
import hashlib
from pyspark.sql import functions as F

# 1. Identify the source file
source_path = Path(
    "/Volumes/workspace/default/raw_landing/yellow_tripdata_2025-01.parquet"
)

assert source_path.is_file(), "Source file missing"

# 2. Calculate its fingerprint
hasher = hashlib.sha256()

with source_path.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b""
    ):
        hasher.update(chunk)

file_hash = hasher.hexdigest()
file_bytes = source_path.stat().st_size

# 3. Read registered versions of January
manifest = spark.table(
    "workspace.control.source_file_manifest"
)

month_versions = (
    manifest
    .filter(
        (F.col("source_type") == "yellow_taxi") &
        (F.col("source_month") == "2025-01")
    )
    .collect()
)

matching = [
    row for row in month_versions
    if row["content_sha256"] == file_hash
    and row["file_bytes"] == file_bytes
]

assert len(matching) == 1, (
    "Source version is not registered exactly once. "
    "Review the manifest before processing."
)

record = matching[0]
status = record["processing_status"]

# 4. Prevent unsafe replacement of a published month
other_published = [
    row for row in month_versions
    if row["processing_status"] == "PUBLISHED"
    and row["content_sha256"] != file_hash
]

if other_published:
    raise RuntimeError(
        "Source content changed for an already published month. "
        "Controlled reprocessing is required."
    )

# 5. Decide whether processing is needed
if status == "PUBLISHED":

    assert record["published_at"] is not None, (
        "Publication timestamp missing"
    )

    assert record["published_run_id"], (
        "Published Job run ID missing"
    )

    required_tables = [
        "workspace.bronze.yellow_taxi_trips",
        "workspace.silver.yellow_taxi_trips",
        "workspace.silver.yellow_taxi_quarantine",
        "workspace.silver.yellow_taxi_out_of_scope",
        "workspace.gold.fact_yellow_trip",
        "workspace.gold.dim_date",
        "workspace.gold.dim_zone",
        "workspace.gold.agg_monthly_kpi",
        "workspace.gold.agg_daily_pickup_zone"
    ]

    missing_tables = [
        table for table in required_tables
        if not spark.catalog.tableExists(table)
    ]

    if missing_tables:
        raise RuntimeError(
            f"Published source has missing tables: {missing_tables}"
        )

    should_process = "false"
    print("SKIP: Source version already published")

elif status == "OBSERVED":
    should_process = "true"
    print("PROCESS: Registered source requires processing")

else:
    raise RuntimeError(
        f"Unsupported manifest status: {status}"
    )

# 6. Send the decision to Databricks Jobs
dbutils.jobs.taskValues.set(
    key="should_process",
    value=should_process
)

print("Source month: 2025-01")
print("Manifest status:", status)
print("Should process:", should_process)
